# Small multiples: the same boundary, different question

A common learning failure is changing the dataset and the cartographic method at the same time. This notebook keeps the real Montreal boundary fixed and changes the question: turnout, winner, and car-share intensity.

The notebook introduces a “map audit” habit: each map must state unit, denominator, classification, source, and audience.

**Reflection questions:** Which of the three maps invites causal overreach? What denominator is missing? Which map would you show to a public audience first, and why?

In [ ]:
# Pyodide/JupyterLite bootstrap: install only pure-Python packages used in this notebook.
import sys, importlib
try:
    import micropip
except Exception:
    micropip = None

async def ensure_packages(packages):
    for pkg, import_name in packages:
        try:
            importlib.import_module(import_name)
        except Exception:
            if micropip is None:
                raise RuntimeError(f'{pkg} is not installed and micropip is unavailable.')
            await micropip.install(pkg)

await ensure_packages([('pandas','pandas'), ('folium','folium'), ('branca','branca'), ('plotly','plotly')])


In [ ]:
from pathlib import Path
import json, math, statistics
import pandas as pd
import folium
from folium.plugins import MarkerCluster, HeatMap, TimestampedGeoJson, MiniMap, Fullscreen, MeasureControl

DATA = Path('../data')

def load_json(name):
    return json.loads((DATA / name).read_text(encoding='utf-8'))

def load_csv(name):
    return pd.read_csv(DATA / name)

def add_standard_controls(m):
    MiniMap(toggle_display=True).add_to(m)
    Fullscreen().add_to(m)
    MeasureControl(primary_length_unit='kilometers').add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

def color_scale(values, colors=('green','orange','red')):
    vals = list(values)
    lo, hi = min(vals), max(vals)
    def pick(v):
        if hi == lo:
            return colors[1]
        t = (v - lo) / (hi - lo)
        return colors[0] if t < .33 else colors[1] if t < .66 else colors[2]
    return pick


In [ ]:
districts = load_json('montreal_districts.geojson')
election = load_csv('election.csv')
carshare = load_csv('carshare.csv')
# Build three maps in tabs by saving HTML snippets.
maps=[]
for column, legend in [('total','Votes cast'), ('Coderre','Candidate Coderre votes'), ('Bergeron','Candidate Bergeron votes')]:
    m = folium.Map(location=[45.52,-73.60], zoom_start=10, tiles='CartoDB positron')
    folium.Choropleth(geo_data=districts, data=election, columns=['district',column], key_on='feature.properties.district', fill_color='PuBuGn', legend_name=legend).add_to(m)
    folium.LayerControl().add_to(m)
    maps.append((legend, m._repr_html_()))
from IPython.display import HTML
tabs = '<h3>Small multiples</h3>' + ''.join([f'<details open><summary>{name}</summary>{html}</details>' for name, html in maps])
HTML(tabs)